In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
import tensorflow as tf
from tensorflow.keras import models, layers


In [28]:
# Load dataset
url = "https://web.stanford.edu/~hastie/ElemStatLearn/datasets/prostate.data"
df = pd.read_csv(url, sep="\t", index_col=0)


df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 97 entries, 1 to 97
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   lcavol   97 non-null     float64
 1   lweight  97 non-null     float64
 2   age      97 non-null     int64  
 3   lbph     97 non-null     float64
 4   svi      97 non-null     int64  
 5   lcp      97 non-null     float64
 6   gleason  97 non-null     int64  
 7   pgg45    97 non-null     int64  
 8   lpsa     97 non-null     float64
 9   train    97 non-null     object 
dtypes: float64(5), int64(4), object(1)
memory usage: 8.3+ KB


In [29]:
# Drop 'train' column used in original book split
df = df.drop(columns=['train'])

# Convert continuous lpsa into binary class: 1 = High PSA, 0 = Low PSA
threshold = df['lpsa'].median()
df['label'] = (df['lpsa'] > threshold).astype(int)

# Drop the original lpsa for classification
X = df.drop(columns=['lpsa', 'label'])  # Features
y = df['label']  # Binary class target

In [22]:
# Normalize features
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)


In [23]:
svm_params = {'C': [0.1, 1, 10], 'kernel': ['linear', 'rbf']}
svm_grid = GridSearchCV(SVC(probability=True), svm_params, cv=5)
svm_grid.fit(X_train, y_train)

print("✅ Best SVM Params:", svm_grid.best_params_)


✅ Best SVM Params: {'C': 10, 'kernel': 'linear'}


In [24]:
rf_params = {'n_estimators': [50, 100], 'max_depth': [None, 5, 10]}
rf_grid = GridSearchCV(RandomForestClassifier(), rf_params, cv=5)
rf_grid.fit(X_train, y_train)

print("✅ Best RF Params:", rf_grid.best_params_)


✅ Best RF Params: {'max_depth': 10, 'n_estimators': 50}


In [25]:
ann_model = models.Sequential([
    layers.Dense(16, activation='relu', input_shape=(X_train.shape[1],)),
    layers.Dense(8, activation='relu'),
    layers.Dense(1, activation='sigmoid')  # binary classification
])

ann_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
ann_model.fit(X_train, y_train, epochs=50, batch_size=8, validation_split=0.2, verbose=0)


C:\Users\Other\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [26]:
def evaluate_model(name, y_true, y_pred):
    print(f"\n📌 {name} Evaluation:")
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print("Precision:", precision_score(y_true, y_pred))
    print("Recall:", recall_score(y_true, y_pred))
    print("F1-Score:", f1_score(y_true, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))

# SVM
svm_pred = svm_grid.predict(X_test)
evaluate_model("SVM", y_test, svm_pred)

# Random Forest
rf_pred = rf_grid.predict(X_test)
evaluate_model("Random Forest", y_test, rf_pred)

# ANN
ann_prob = ann_model.predict(X_test).flatten()
ann_pred = (ann_prob > 0.5).astype(int)
evaluate_model("ANN", y_test, ann_pred)



📌 SVM Evaluation:
Accuracy: 0.8
Precision: 0.7
Recall: 0.875
F1-Score: 0.7777777777777778
Confusion Matrix:
 [[9 3]
 [1 7]]

📌 Random Forest Evaluation:
Accuracy: 0.8
Precision: 0.7
Recall: 0.875
F1-Score: 0.7777777777777778
Confusion Matrix:
 [[9 3]
 [1 7]]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 166ms/step

📌 ANN Evaluation:
Accuracy: 0.8
Precision: 0.7
Recall: 0.875
F1-Score: 0.7777777777777778
Confusion Matrix:
 [[9 3]
 [1 7]]


In [27]:
import joblib
joblib.dump(svm_grid.best_estimator_, '../models/svm_model.pkl')
joblib.dump(rf_grid.best_estimator_, '../models/rf_model.pkl')
model.save('../models/ann_model.keras')


NameError: name 'model' is not defined